This notebook demonstrates:
- Part 1: how to create a polynomial neural network using ``PolynomialNetwork`` class, 
- Part 2: how to extract the polynomial coefficients and monomials from the network using ``polynomial_nn_expansion`` function, and
- Part 3: how to use ``pyhpc`` package https://arxiv.org/pdf/1907.00096 , https://homepages.math.uic.edu/~jan/phcpy_doc_html/index.html, to solve polynomial systems (using homotopy continuation method). GPU support ? https://homepages.math.uic.edu/~jan/Talks/siampp16.pdf

In [1]:
# Import standard libraries
import sys

sys.path.append("..")

import phcpy  # Homotopy continuation package
from phcpy.dimension import get_core_count # For getting number of CPU cores
from phcpy.solver import solve # For solving polynomial systems
from phcpy.solutions import filter_real # For filtering real solutions

from sympy.plotting import plot_implicit # For plotting implicit curves
import matplotlib.pyplot as plt # For plotting

# Import our modules
from src.model import *
from src.nnexpansion import *

PHCv2.4.90 released 2024-03-20 works!


ModuleNotFoundError: No module named 'src.model'

### Part 1
Create a (non-homogeneous) polynomial neural network using `PolynomialNetwork` class. The class is defined in `model.py` file. There are four arguments to initialize the class:
   - `input_dim`: input dimension
   - `output_dim`: output dimension
   - `hidden_dims`: a list of hidden layer dimensions
   - `polynomial_degree`: polynomial degree of the activation function
  
The parameters of the activation functions are trainable.

In the following example, we create a polynomial neural network with input dimension $2$, output dimension $2$, two hidden layers with $3$ neurons each, and polynomial degree $2$.

In [ ]:
# Specify network parameters
input_dim = 2
output_dim = 2
hidden_dims = [3, 3]
polynomial_degree = 2

# Initialize the network
model = PolyNetwork(
    input_dim=input_dim,
    output_dim=output_dim,
    hidden_dims=hidden_dims,
    polynomial_degree=polynomial_degree,
)

# Print the model architecture
print(model)

# Print the total number of parameters
total_params = sum(p.numel() for p in model.parameters())
print("Total number of parameters:", total_params)

# Test the polynomial neural network, batch size 5
x_test = torch.randn(5, input_dim)
y_test = model(x_test)
print("Model output:", y_test)

### Part 2
In this part, we use `polynomial_nn_expansion` function to extract the polynomial coefficients and monomials from the polynomial neural network created in Part 1. The function is defined in `nnexpansion.py` file. The function takes a `PolynomialNetwork` object as input and returns two objects:
   - `coefficients`: a 2d numpy arrays, where each array contains the coefficients of the polynomial expansion for each output dimension. Each row corresponds to an output dimension, and each column corresponds to a monomial.
   - `monomials`: a list of tuples, where each inner tuple contains the monomials corresponding to the coefficients in the `coefficients` list.


In [ ]:
# Monomial Expansion
polys, monoms, C = polynomial_nn_expansion(model)
print("Number of monomials: ", len(monoms))
print("Shape of Coefficient matrix:", C.shape)

print("Monomials:", monoms)
print("Polynomials:", polys)
# print("Coefficient matrix:\n", np.round(C, 4))

We then verify the polynomial expansion is correct by comparing the output of the polynomial neural network and the output of the polynomial expansion on random inputs. There are five arguments to the `verify_expansion` function:
   - `model`: a `PolynomialNetwork` object
   - `monoms`: a list of tuples, where each inner tuple contains the monomials corresponding to the coefficients in the `coefficients` list.
   - `C`: a 2d numpy arrays, where each array contains the coefficients of the polynomial expansion for each output dimension. Each row corresponds to an output dimension, and each column corresponds to a monomial.
   - `test_inputs`: (optional) a torch tensor of shape (n_test, input_dim), where n_test is the number of test inputs. If not provided, random inputs will be generated.
   - `n_test`: (optional) number of test inputs to generate if `test_inputs` is not provided. Default is 100.
   - `tolerance`: (optional) tolerance for comparing the outputs. Default is 1e-6.

In [ ]:
# Verify the expansion is correct
print("\n" + "=" * 50)
is_correct = verify_expansion(model, monoms, C, n_test = 100, tolerance = 1e-6)
print(f"Expansion verification: {'PASSED' if is_correct else 'FAILED'}")

``compute_class_differences`` function is used to compute the differences between the outputs of the polynomial neural network and the polynomial expansion. The function takes four arguments:
   - `C`: a 2d numpy arrays, where each array contains the coefficients of the polynomial expansion for each output dimension. Each row corresponds to an output dimension, and each column corresponds to a monomial.
   - `gold_class`: an integer, the index of the gold class (ground truth class).

The output is a numpy array of shape ``(output_dim-1, num_monomials)``, where each element is the difference between the output of the polynomial neural network and the polynomial expansion for each output dimension.

In [ ]:
# Test the compute_class_differences function
print("Testing compute_class_differences function:")
print(f"Coefficient matrix shape: {C.shape}")

# Test with gold_class = 0
gold_class = 0
coeff_diff = compute_class_differences(C, gold_class)
print(f"Shape of coefficient differences: {coeff_diff.shape}")

### Part 3

We show some examples of using `pyhpc` package to solve polynomial systems. Installation instructions can be found at https://github.com/janverschelde/PHCpack.

On Ubuntu/Debian, ``pyhpc`` is pre-built and available via the package manager. Thus, we can install it system-wide and then copy the necessary files to our virtual environment. For example, one can easily install `pyhpc` in your virtual environment `venv` using the following commands:

```bash
sudo apt install python3-phcpy # Install phcpy system-wide
source venv/bin/activate       # Activate your virtual environment

SITE=$(python -c 'import sysconfig; print(sysconfig.get_paths()["purelib"])') # Get the site-packages directory of the virtual environment
cp /usr/lib/python3/dist-packages/libPHCpack.cpython-312-x86_64-linux-gnu.so "$SITE/" # Copy the shared library to the site-packages directory
cp -r /usr/lib/python3/dist-packages/phcpy "$SITE/" # Copy the phcpy package to the site-packages directory
```

In the following example, we solve a polynomial system with two equations and two variables. The equations represent the intersection of two curves: a circle and a cubic curve. We use ``phcpy.solver.solve`` to find all complex solutions, and then filter the real solutions using ``phcpy.solutions.filter_real``.

In [ ]:
# Example: Solving a 2D Polynomial System
# Let's create two nontrivial curves and find their intersection points

# Define the polynomial system: intersection of an ellipse and a cubic curve
# Curve 1: Ellipse  x^2/4 + y^2/9 = 1  =>  9*x^2 + 4*y^2 = 36
# Curve 2: Cubic    y = x^3 - 2*x      =>  x^3 - 2*x - y = 0

polynomial_system = [
    "9*x^2 + 4*y^2 - 36;",    # Ellipse: x^2/4 + y^2/9 = 1
    "x^3 - 2*x - y;"          # Cubic: y = x^3 - 2*x
]

print("Polynomial System:")
print("Curve 1 (Ellipse): 9x² + 4y² = 36")  
print("Curve 2 (Cubic):   y = x³ - 2x")
print("\nSolving system using PHCpy...")

# Solve the polynomial system with multiple threads
num_cores = get_core_count() - 1 # Leave one core free
print(f"Using {num_cores} CPU cores for computation.")
wstart = time.time()
solutions = solve(polynomial_system, tasks=num_cores)
wend = time.time()
print(f"Solving the system took {wend - wstart:.5f} seconds. Total number of complex solutions: {len(solutions)}")

# Filter real solutions
real_solutions = filter_real(solutions, tol=1.0e-10, oper='select')
print(f"Number of real solutions: {len(real_solutions)}")

# Extract real solution coordinates
real_points = []
for sol in real_solutions:
    # Parse solution string to extract x, y coordinates
    lines = sol.strip().split('\n')
    x_line = [line for line in lines if line.strip().startswith('x :')][0]
    y_line = [line for line in lines if line.strip().startswith('y :')][0]
    
    # Extract real parts (first number after ':')
    x_val = float(x_line.split(':')[1].split()[0])
    y_val = float(y_line.split(':')[1].split()[0])
    real_points.append((x_val, y_val))
    
print("\nReal intersection points:")
for i, (x, y) in enumerate(real_points):
    print(f"Point {i+1}: x = {x:.6f}, y = {y:.6f}")

In [ ]:
# Visualization: Plot curves and intersection points
# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Plot curves using implicit method
x_vals = np.linspace(-3, 3, 1000)

# Ellipse: x²/4 + y²/9 = 1 => y = ±3√(1 - x²/4)
x_ell = np.linspace(-2, 2, 500)
y_ell_pos = 3 * np.sqrt(1 - x_ell**2/4)
y_ell_neg = -3 * np.sqrt(1 - x_ell**2/4)

# Cubic: y = x³ - 2x
y_cubic = x_vals**3 - 2*x_vals

# Plot curves
ax.plot(x_ell, y_ell_pos, 'b-', linewidth=2, label='Ellipse: x²/4 + y²/9 = 1')
ax.plot(x_ell, y_ell_neg, 'b-', linewidth=2)
ax.plot(x_vals, y_cubic, 'r-', linewidth=2, label='Cubic: y = x³ - 2x')

# Plot intersection points
if real_points:
    x_pts, y_pts = zip(*real_points)
    ax.scatter(x_pts, y_pts, c='green', s=100, zorder=5, 
               label=f'Intersections ({len(real_points)} points)')
    
    # Annotate points
    for i, (x, y) in enumerate(real_points):
        ax.annotate(f'P{i+1}', (x, y), xytext=(5, 5), 
                   textcoords='offset points', fontsize=10)

ax.set_xlim(-3, 3)
ax.set_ylim(-4, 4)
ax.grid(True, alpha=0.3)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Polynomial System: Ellipse-Cubic Intersection')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

# Verify solutions
print("\nSolution verification:")
for i, (x, y) in enumerate(real_points):
    ellipse_val = 9*x**2 + 4*y**2
    cubic_val = x**3 - 2*x - y
    print(f"P{i+1}: ({x:.6f}, {y:.6f}) - Ellipse: {ellipse_val:.6f}, Cubic: {cubic_val:.2e}")